In [ ]:
library(tidyverse)
library(ggplot2)
library(easystats)

In [ ]:
files <- list.files(path = "D:/BNC Full Data/12-9_5PM Run/CSV",
                    pattern = "\\.csv$",
                    full.names = TRUE)

df_full <- read_csv(files, id = "file_name")

# df_full <- readRDS("D:/BNC Full Data/12-9_5PM Run/12-9_5PM_Full-Data.rds")

df_full <- df_full %>% 
    arrange(Sentence_ID) %>% # Removes duplicates
    group_by(Sentence_Text) %>%
    mutate(first_Sentence_ID = first(Sentence_ID)) %>%
    filter(Sentence_ID == first_Sentence_ID) %>%
    ungroup() %>%
    select(-first_Sentence_ID) 

df_full <- df_full %>% #Tags first word in an NP
    arrange(Sentence_ID, Word_Token_Index) %>% 
    group_by(Sentence_ID) %>% 
    mutate(
        prev_is_NP = lag(Is_NP, default = FALSE),
        prev_NP_Head_Text = lag(NP_Head_Text),

        first_token_of_NP = Is_NP & (!prev_is_NP | NP_Head_Text != prev_NP_Head_Text)
    ) %>% 
    ungroup() %>% 
    select(-prev_is_NP, -prev_NP_Head_Text)

df_full <- df_full %>% #Propogates index of first NP down to the full phrase
    group_by(Sentence_ID, np_id = consecutive_id(phrase_token)) %>% 
    mutate(
        np_start_idx = ifelse(
            is.na (phrase_token), 
            NA,
            min(token_index)
        )
    ) %>% 
    ungroup() %>% 
    select(-np_id)

df_full <- df_full %>% #Renames and sets baseline vals for analysis
    mutate(
        definiteness = factor(NP_Definiteness,
        levels = c("indefinite", "definite"),
        labels = c("indef", "def"))
    ) %>% 
    mutate(
        argPos = factor(
            NP_Argument,
            levels = c("dir_object", "subject"),
            labels = c("obj", "sbj")
        )
    ) %>% 
    mutate(surprisal = Phrase_Surprisal)

write_rds(df_full, "D:/BNC Full Data/12-9_5PM Run/12-9_5PM_Full-Data.rds")

In [ ]:
df <- df_full %>% 
    arrange(Sentence_ID) %>% # Removes duplicates
        group_by(Sentence_Text) %>%
        mutate(first_Sentence_ID = first(Sentence_ID)) %>%
        filter(Sentence_ID == first_Sentence_ID) %>%
        ungroup() %>%
        select(-first_Sentence_ID) %>% 
    filter(Is_NP == TRUE, # Filtering Criteria
            Is_Head_Noun == TRUE,
            Modality == "written", 
            Sent_Verb_Count == 1,
            Sent_Auxiliary_Count == 0,
            Sent_Subject_Count == 1,
            Sent_Tot_Obj_Count %in% 1,
            Sent_Dir_Object_Count == 1 ,
            Sent_Ind_Object_Count == 0,
            Sent_Sub_Conj_Count == 0,
            Sent_Coord_Conj_Count == 0, 
            Clausal_Complement_Count == 0,
            Sent_Relative_Clause_Count == 0, 
            Sent_Adv_Clause_Count == 0, 
            Sent_Prep_Phrase_Count == 0,
            Sent_Comma_Count == 0,
            !str_detect(Sentence_Text, "\\?"),
            NP_Definiteness  %in% c("definite", "indefinite"),
            NP_Argument %in% c("subject", "dir_object"),
            Sent_Transitive == TRUE,
            ) %>%
            drop_na(Phrase_Surprisal) %>% # Drops values w/out valid surprisal value
            group_by(Sentence_ID) %>% # Drops sentences without one subject and one object
                filter(n() == 2 & n_distinct(NP_Argument) == 2) %>%
                ungroup()

df <- df %>% 
        select(Sentence_ID, Sentence_Text, Phrase_Token, surprisal, definiteness, argPos, Word_Token_Index, within_file_id, within_chunk_id, np_start_idx)

saveRDS(df, file = "Results 12-9/filtered_12-9.rds")
write_csv(df, "Results 12-9/filtered_12-9.csv")